In [0]:
# Config
from pyspark.sql import functions as F

catalog = "endtoend"
schema = "iras_data"

In [0]:
# Read the 4 source tables
df_1 = spark.table(f"{catalog}.{schema}.assessable_income_of_individuals")
df_2 = spark.table(f"{catalog}.{schema}.assessable_income_of_individuals_by_tax_group")
df_3 = spark.table(f"{catalog}.{schema}.income_of_individuals_by_income_type")
df_4 = spark.table(f"{catalog}.{schema}.income_of_individuals_by_income_type_and_tax_group")

In [0]:
# Bronze: raw data unchanged, plus audit columns
bronze_tables = {
    "bronze_assessable_income": ("assessable_income_of_individuals", df_1),
    "bronze_assessable_income_by_group": ("assessable_income_of_individuals_by_tax_group", df_2),
    "bronze_income_by_type": ("income_of_individuals_by_income_type", df_3),
    "bronze_income_by_type_group": ("income_of_individuals_by_income_type_and_tax_group", df_4),
}

for bronze_name, (source_name, df) in bronze_tables.items():
    (df.withColumn("ingested_at", F.current_timestamp())
       .withColumn("source_table", F.lit(f"{catalog}.{schema}.{source_name}"))
       .write.mode("overwrite")
       .saveAsTable(f"{catalog}.{schema}.{bronze_name}"))
    print(f"Wrote {catalog}.{schema}.{bronze_name}")

In [0]:
# Validation: row counts should match the source files
expected = {
    "bronze_assessable_income": 22,
    "bronze_assessable_income_by_group": 44,
    "bronze_income_by_type": 154,
    "bronze_income_by_type_group": 308,
}

for name, exp in expected.items():
    n = spark.table(f"{catalog}.{schema}.{name}").count()
    print(f"{'✅' if n == exp else '❌'} {name}: {n} rows (expected {exp})")


In [0]:
for name in ["bronze_assessable_income",
             "bronze_assessable_income_by_group",
             "bronze_income_by_type",
             "bronze_income_by_type_group"]:
    print(name)
    display(spark.table(f"{catalog}.{schema}.{name}").limit(5))